# tunedGNN-style OGBN-Arxiv with tf_gnns

This notebook reproduces the tunedGNN `large_graph/main-arxiv.py` architecture and training recipe using `tf_gnns` + TensorFlow backend.


In [ ]:
# If needed:
# %pip install -U tf_gnns ogb keras tensorflow

In [2]:
from ogb.nodeproppred import NodePropPredDataset


/home/charilaos/Workspace/tf_gnns/.venv/lib/python3.11/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/charilaos/Workspace/tf_gnns/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import tensorflow as tf
import keras
from tf_gnns.models import GCNv2

seed = 42
np.random.seed(seed)
keras.utils.set_random_seed(seed)


I0000 00:00:1780333832.321001   54702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780333832.350256   54702 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780333833.187541   54702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
dataset = NodePropPredDataset(name="ogbn-arxiv", root="dataset")
split_idx = dataset.get_idx_split()
graph, labels = dataset[0]

x = graph["node_feat"].astype(np.float32)
edge_index = graph["edge_index"].astype(np.int32)
y = labels.reshape(-1).astype(np.int32)

num_nodes = int(x.shape[0])
num_classes = int(y.max()) + 1

train_idx = split_idx["train"].astype(np.int32).reshape(-1)
valid_idx = split_idx["valid"].astype(np.int32).reshape(-1)
test_idx = split_idx["test"].astype(np.int32).reshape(-1)

print(f"num_nodes={num_nodes:,}, num_edges={edge_index.shape[1]:,}, num_classes={num_classes}")
print(f"train/valid/test = {len(train_idx):,}/{len(valid_idx):,}/{len(test_idx):,}")


num_nodes=169,343, num_edges=1,166,243, num_classes=40
train/valid/test = 90,941/29,799/48,603


/home/charilaos/Workspace/tf_gnns/.venv/lib/python3.11/site-packages/ogb/nodeproppred/dataset.py:70: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  loaded_dict = torch.load(pre_processed_file_path)


In [5]:
# Match tunedGNN preprocessing in main-arxiv.py:
# to_undirected -> remove_self_loops -> add_self_loops
senders = edge_index[0]
receivers = edge_index[1]

rev_senders = receivers
rev_receivers = senders
senders = np.concatenate([senders, rev_senders], axis=0)
receivers = np.concatenate([receivers, rev_receivers], axis=0)

non_self = senders != receivers
senders = senders[non_self]
receivers = receivers[non_self]

self_nodes = np.arange(num_nodes, dtype=np.int32)
senders = np.concatenate([senders, self_nodes], axis=0)
receivers = np.concatenate([receivers, self_nodes], axis=0)

num_edges = int(senders.shape[0])
print(f"processed edges={num_edges:,}")


processed edges=2,501,829


In [6]:
td = {
    "nodes": keras.ops.convert_to_tensor(x, dtype="float32"),
    "edges": keras.ops.zeros((num_edges, 1), dtype="float32"),
    "senders": keras.ops.convert_to_tensor(senders, dtype="int32"),
    "receivers": keras.ops.convert_to_tensor(receivers, dtype="int32"),
    "n_edges": keras.ops.convert_to_tensor([num_edges], dtype="int32"),
    "n_nodes": keras.ops.convert_to_tensor([num_nodes], dtype="int32"),
    "n_graphs": keras.ops.convert_to_tensor(1, dtype="int32"),
}

train_idx_t = keras.ops.convert_to_tensor(train_idx, dtype="int32")
valid_idx_t = keras.ops.convert_to_tensor(valid_idx, dtype="int32")
test_idx_t = keras.ops.convert_to_tensor(test_idx, dtype="int32")
y_t = keras.ops.convert_to_tensor(y, dtype="int32")


I0000 00:00:1780333841.279055   54702 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21453 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


In [7]:
# tunedGNN-like architecture (large_graph/lg_model.py):
# hidden=256, local_layers=7, in_dropout=0.15, dropout=0.5,
# residual linear skip, batchnorm, ReLU, no JK by default.
model = GCNv2(
    hidden_units=256,
    output_units=num_classes,
    num_layers=7,
    add_self_loops=False,
    input_dropout_rate=0.15,
    dropout_rate=0.5,
    residual=True,
    residual_projection=True,
    batchnorm=True,
    layernorm=False,
    use_shortcut=False,
    use_bias=True,
)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
weight_decay = 5e-4


In [8]:
def accuracy_on_indices(logits, indices, labels):
    pred = keras.ops.argmax(keras.ops.take(logits, indices, axis=0), axis=-1)
    truth = keras.ops.take(labels, indices, axis=0)
    return float(keras.ops.mean(keras.ops.cast(pred == truth, "float32")))

@tf.function
def train_step(td, train_idx_t, y_t):
    with tf.GradientTape() as tape:
        logits = model(td, training=True)["nodes"]
        train_logits = keras.ops.take(logits, train_idx_t, axis=0)
        train_labels = keras.ops.take(y_t, train_idx_t, axis=0)
        ce = loss_fn(train_labels, train_logits)
        #l2 = keras.ops.sum(
        #    keras.ops.stack([keras.ops.sum(v * v) for v in model.trainable_variables])
        #)
        loss = ce #+ 0.5 * weight_decay * l2

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def eval_all(td, y_t):
    logits = model(td, training=False)["nodes"]
    tr = accuracy_on_indices(logits, train_idx_t, y_t)
    va = accuracy_on_indices(logits, valid_idx_t, y_t)
    te = accuracy_on_indices(logits, test_idx_t, y_t)
    return tr, va, te


In [9]:
epochs = 2000
display_step = 10
patience = 200

best_val = -1.0
best_test = -1.0
best_epoch = -1

for epoch in range(1, epochs + 1):
    loss = train_step(td, train_idx_t, y_t)
    train_acc, val_acc, test_acc = eval_all(td, y_t)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc
        best_epoch = epoch

    if epoch % display_step == 0 or epoch == 1:
        print(
            f"Epoch {epoch:04d} | loss={loss:.4f} | "
            f"train={100*train_acc:.2f}% val={100*val_acc:.2f}% test={100*test_acc:.2f}% | "
            f"best_val={100*best_val:.2f}% best_test={100*best_test:.2f}% (ep {best_epoch})"
        )

    if epoch - best_epoch >= patience:
        print(f"Early stop at epoch {epoch} (patience={patience}).")
        break

print(f"Final best: val={100*best_val:.2f}% test={100*best_test:.2f}% at epoch {best_epoch}")


Epoch 0001 | loss=32.8560 | train=17.05% val=20.59% test=17.55% | best_val=20.59% best_test=17.55% (ep 1)
Epoch 0010 | loss=10.0095 | train=23.70% val=26.71% test=24.62% | best_val=32.41% best_test=28.84% (ep 6)
Epoch 0020 | loss=4.9414 | train=25.57% val=24.71% test=27.78% | best_val=32.41% best_test=28.84% (ep 6)
Epoch 0030 | loss=3.3356 | train=32.05% val=30.11% test=34.38% | best_val=32.41% best_test=28.84% (ep 6)
Epoch 0040 | loss=2.6900 | train=46.79% val=49.19% test=50.66% | best_val=49.19% best_test=50.66% (ep 40)
Epoch 0050 | loss=2.3965 | train=53.93% val=56.16% test=56.52% | best_val=56.16% best_test=56.52% (ep 50)
Epoch 0060 | loss=2.1850 | train=58.30% val=60.24% test=60.47% | best_val=60.24% best_test=60.47% (ep 60)
Epoch 0070 | loss=2.0188 | train=60.48% val=62.16% test=61.69% | best_val=62.23% best_test=61.83% (ep 67)
Epoch 0080 | loss=1.9254 | train=61.84% val=63.08% test=61.94% | best_val=63.08% best_test=61.94% (ep 80)
Epoch 0090 | loss=1.8339 | train=63.00% val=64.0

Reported max performance for this architecture: 

| Split  | Accuracy (reported SOTA GCN) | Accuracy (this notebook) | % Diff. to SOTA GCN | 
|---|----|---------|---------------------|
| Validation | 74.47% | 73.98% | -0.6% | 
| Test | 73.60% | 72.88% | -0.1% |